In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

IN_PATH = "allegations_202007271729.csv"

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "axes.spines.top": False,
    "axes.spines.right": False,
})
INK = "#2b2b2b"
MUTED = "#6b6b6b"
GRID = "#ececec"


In [3]:
# ------------------------------------------------------------------ load and clean
df = pd.read_csv("dsc106_project2_dataset.csv")
df = df[["unique_mos_id", "complaint_id", "year_received", "month_received",
         "rank_incident", "rank_now", "board_disposition"]].dropna()  # no drop_duplicates: each row is an allegation
df["substantiated"] = df["board_disposition"].str.startswith("Substantiated")
df = df.sort_values(["unique_mos_id", "year_received", "month_received", "complaint_id"])

# Rank scales.
# ANY    : Detective counts as one step above Police Officer (used by Visualization 1)
# LADDER : Detective counts the same as Police Officer (used by Visualization 2)
ANY = {"Police Officer": 0, "Detective": 1, "Sergeant": 2, "Lieutenant": 3,
       "Captain": 4, "Deputy Inspector": 5, "Inspector": 6, "Chiefs and other ranks": 7}
LADDER = {"Police Officer": 0, "Detective": 0, "Sergeant": 1, "Lieutenant": 2,
          "Captain": 3, "Deputy Inspector": 4, "Inspector": 5, "Chiefs and other ranks": 6}

# ------------------------------------------------------------------ officer table
first = df.groupby("unique_mos_id").first()
off = pd.DataFrame({
    "first_year": first["year_received"],
    "rank_first": first["rank_incident"],
    "rank_now": first["rank_now"],
})
off["promoted_any"] = off["rank_now"].map(ANY) > off["rank_first"].map(ANY)
off["promoted_ladder"] = off["rank_now"].map(LADDER) > off["rank_first"].map(LADDER)

sub = df[df["substantiated"]]
off["n_sub_allegations"] = sub.groupby("unique_mos_id").size()
off["n_sub_complaints"] = sub.groupby("unique_mos_id")["complaint_id"].nunique()
off = off.dropna(subset=["n_sub_allegations", "n_sub_complaints"])
off["cohort"] = (off["first_year"] // 5 * 5).astype(int)
N = len(off)

In [6]:
# ================================================================== VISUALIZATION 1
# Proposition is TRUE: more substantiated misconduct goes with more promotion.
bins = [0, 1, 2, 3, 5, 10_000]
labels = ["1", "2", "3", "4-5", "6+"]
off["alleg_group"] = pd.cut(off["n_sub_allegations"], bins, labels=labels)
v1 = (off.groupby("alleg_group", observed=True)["promoted_any"]
         .agg(rate="mean", n="size").reset_index())

fig, ax = plt.subplots(figsize=(9.5, 6))
x = np.arange(len(v1))
y = v1["rate"].values * 100

# thin connecting line plus dots, so the axis starts at 40% without bars that encode length
ax.plot(x, y, color="#b4182a", lw=2.4, zorder=2, alpha=0.9)
ax.scatter(x, y, s=330, color="#b4182a", zorder=3, edgecolor="white", linewidth=2)
for xi, yi in zip(x, y):
    ax.text(xi, yi + 2.3, f"{yi:.0f}%", ha="center", va="bottom",
            fontsize=15, fontweight="bold", color=INK)

ax.set_xticks(x)
ax.set_xticklabels(v1["alleg_group"], fontsize=13, color=INK)
ax.set_ylim(40, 78)
ax.set_yticks([40, 50, 60, 70])
ax.set_yticklabels([f"{t}%" for t in [40, 50, 60, 70]], fontsize=12, color=MUTED)
ax.yaxis.grid(True, color=GRID)
ax.set_axisbelow(True)
ax.tick_params(length=0)
ax.set_xlabel("Substantiated misconduct allegations against the officer",
              fontsize=12, color=INK, labelpad=10)
ax.set_ylabel("Share of officers promoted", fontsize=12, color=INK, labelpad=10)

# annotation drawing the eye to the gap
ax.annotate("", xy=(4.28, y[4]), xytext=(4.28, y[0]),
            arrowprops=dict(arrowstyle="<->", color=MUTED, lw=1.4))
ax.text(4.36, (y[0] + y[4]) / 2, f"+{y[4]-y[0]:.0f}\npoints", fontsize=12,
        color=MUTED, va="center", ha="left")
ax.set_xlim(-0.5, 5.0)

fig.text(0.06, 0.945, "More substantiated allegations, more promotions",
         fontsize=20, fontweight="bold", color=INK, ha="left")
fig.text(0.06, 0.895, "Share of NYPD officers promoted, by number of substantiated allegations on record",
         fontsize=13, color=MUTED, ha="left")
fig.text(0.06, 0.045,
         f"Source: NYPD CCRB allegations, through July 2020 (n = {N:,} officers).",
         fontsize=9, color=MUTED, ha="left")
fig.text(0.06, 0.02,
         "Promoted = current rank is above rank at first complaint, including a move to Detective.",
         fontsize=9, color=MUTED, ha="left")
fig.subplots_adjust(top=0.82, bottom=0.19, left=0.09, right=0.96)
fig.savefig("viz1_proposition_true.png", dpi=200)
plt.close(fig)

In [7]:
# ================================================================== VISUALIZATION 2
# Proposition is FALSE: tenure, not misconduct, explains who gets promoted.
off["complaint_group"] = pd.cut(off["n_sub_complaints"], [0, 1, 2, 10_000],
                                labels=["1 complaint", "2 complaints", "3+ complaints"])
v2 = (off.groupby(["cohort", "complaint_group"], observed=True)["promoted_ladder"]
         .agg(rate="mean", n="size").reset_index())
cohorts = sorted(off["cohort"].unique())
cohort_labels = [f"{c}-{str(c + 4)[-2:]}" for c in cohorts]

colors = {"1 complaint": "#9ecae1", "2 complaints": "#3182bd", "3+ complaints": "#08306b"}
offsets = {"1 complaint": -0.2, "2 complaints": 0.0, "3+ complaints": 0.2}

fig, ax = plt.subplots(figsize=(9.5, 6))
for g, c in colors.items():
    d = v2[v2["complaint_group"] == g]
    xs = [cohorts.index(k) + offsets[g] for k in d["cohort"]]
    solid = d["n"] >= 30
    ax.scatter(np.array(xs)[solid], d["rate"][solid] * 100, s=140, color=c,
               zorder=3, edgecolor="white", linewidth=1.5, label=g)
    # hollow markers where fewer than 30 officers, so thin groups don't look as solid
    ax.scatter(np.array(xs)[~solid], d["rate"][~solid] * 100, s=140, facecolor="white",
               edgecolor=c, linewidth=2, zorder=3)

# bracket showing the spread across complaint counts within one cohort vs. across cohorts
by_cohort = v2.groupby("cohort")["rate"].agg(["min", "max"]) * 100
for i, c in enumerate(cohorts):
    ax.plot([i, i], [by_cohort.loc[c, "min"], by_cohort.loc[c, "max"]],
            color="#bdbdbd", lw=6, solid_capstyle="round", zorder=1, alpha=0.7)

ax.set_xticks(range(len(cohorts)))
ax.set_xticklabels(cohort_labels, fontsize=12, color=INK)
ax.set_ylim(0, 100)
ax.set_yticks(range(0, 101, 20))
ax.set_yticklabels([f"{t}%" for t in range(0, 101, 20)], fontsize=12, color=MUTED)
ax.yaxis.grid(True, color=GRID)
ax.set_axisbelow(True)
ax.tick_params(length=0)
ax.set_xlabel("Year of officer's first complaint (5-year groups)", fontsize=12,
              color=INK, labelpad=10)
ax.set_ylabel("Share of officers promoted", fontsize=12, color=INK, labelpad=10)

ax.annotate("", xy=(0.0, 18), xytext=(0.0, 18))
ax.text(2.55, 90, "Across eras, the drop is about 70 points", fontsize=12,
        color=INK, fontweight="bold", ha="left")
ax.text(2.55, 83, "Within an era, 3+ complaints add roughly 10-15", fontsize=12,
        color=MUTED, ha="left")
ax.text(2.55, 77.5, "points over a single complaint", fontsize=12, color=MUTED, ha="left")

handles = [Line2D([0], [0], marker="o", color="w", markerfacecolor=c, markersize=10, label=g)
           for g, c in colors.items()]
handles.append(Line2D([0], [0], marker="o", color="w", markerfacecolor="white",
                      markeredgecolor=MUTED, markersize=10, label="fewer than 30 officers"))
ax.legend(handles=handles, loc="lower left", frameon=False, fontsize=10,
          bbox_to_anchor=(0.01, 0.02))

fig.text(0.06, 0.945, "Era matters far more than misconduct in who gets promoted",
         fontsize=18, fontweight="bold", color=INK, ha="left")
fig.text(0.06, 0.895, "Share of NYPD officers promoted, by era of first complaint and number of substantiated complaints",
         fontsize=12, color=MUTED, ha="left")
fig.text(0.06, 0.045,
         f"Source: NYPD CCRB allegations, through July 2020 (n = {N:,} officers).",
         fontsize=9, color=MUTED, ha="left")
fig.text(0.06, 0.02,
         "Promoted = moved up the rank ladder (Detective not counted).",
         fontsize=9, color=MUTED, ha="left")
fig.subplots_adjust(top=0.82, bottom=0.19, left=0.09, right=0.96)
fig.savefig("viz2_proposition_false.png", dpi=200)
plt.close(fig)

print(v2.assign(rate=(v2["rate"] * 100).round(0)).pivot(
    index="cohort", columns="complaint_group", values=["rate", "n"]).to_string())

                       rate                                      n                           
complaint_group 1 complaint 2 complaints 3+ complaints 1 complaint 2 complaints 3+ complaints
cohort                                                                                       
1985                   74.0         75.0          80.0        19.0          4.0           5.0
1990                   52.0         67.0          64.0        58.0         18.0          11.0
1995                   46.0         47.0          62.0       175.0         45.0          21.0
2000                   43.0         44.0          60.0       439.0         87.0          52.0
2005                   31.0         35.0          46.0       910.0        212.0          80.0
2010                   24.0         24.0          31.0       778.0        144.0          39.0
2015                    5.0          1.0           0.0       809.0         78.0          12.0
